# CBMC, NuSMV & friends — quickstart

These are command-line tools; we drive them from notebook cells with `!`.

- **Codespaces:** `cbmc`, `cryptol`, `saw`, `NuSMV` are all preinstalled.
- **Colab:** the setup cell installs `cbmc` and `NuSMV` (seconds). `cryptol`/`saw`
  are large downloads — use Codespaces for those.

In [ ]:
# Setup — installs each tool only if missing (a no-op in Codespaces).
import shutil, subprocess

def have(cmd):
    return shutil.which(cmd) is not None

if not have('cbmc'):
    print('Installing cbmc ...')
    subprocess.run('apt-get -qq update && apt-get -qq install -y cbmc', shell=True)

if not have('NuSMV'):
    print('Installing NuSMV ...')
    subprocess.run(
        'wget -q https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz -O /tmp/nusmv.tgz '
        '&& mkdir -p /opt/nusmv && tar -xzf /tmp/nusmv.tgz -C /opt/nusmv --strip-components=1 '
        '&& ln -sf /opt/nusmv/bin/NuSMV /usr/local/bin/NuSMV',
        shell=True)

for c in ('cbmc', 'NuSMV', 'cryptol', 'saw', 'z3'):
    print(f'{c:8s}: ' + (shutil.which(c) or 'not found (use Codespaces)'))

## 1. CBMC — bounded model checking of C

Write a tiny C program with an out-of-bounds access, then let CBMC find it.

In [ ]:
%%writefile bounds.c
int main(void) {
    int a[5];
    int i;
    __CPROVER_assume(i >= 0 && i <= 5);   // note: index 5 is out of bounds for a[5]
    a[i] = 42;                            // --bounds-check flags i == 5
    return 0;
}

In [ ]:
# CBMC should report VERIFICATION FAILED with a counterexample (i = 5).
!cbmc bounds.c --bounds-check

## 2. NuSMV — model checking an SMV model

A one-bit toggle: `b` flips each step. Both specs below should be **true**.

In [ ]:
%%writefile toggle.smv
MODULE main
VAR
  b : boolean;
ASSIGN
  init(b) := FALSE;
  next(b) := !b;
SPEC AG (b -> AX !b)   -- whenever b holds, next step it does not
LTLSPEC G F b          -- b is true infinitely often

In [ ]:
# Look for '-- specification ... is true' in the output.
!NuSMV toggle.smv

## 3. Cryptol / SAW

Preinstalled in Codespaces (large to install in Colab). Quick check:

In [ ]:
!cryptol --version || echo 'cryptol not found here — open this in Codespaces'

## nuXmv & visualization

`nuXmv` is license-gated, so it is not in the public image. For nuXmv plus
state-graph / BDD visualization — and to upload your own `.smv` files — use the
**smvis web app**: <https://smvis-378135919048.us-central1.run.app>

For finite-state models, `NuSMV` (above) accepts the same SMV language and
prints the same `is true/false` verdicts.